# Data Preparation — Common Pipeline (Steps 1-7)

Acquires raw mushroom images, validates, deduplicates, splits, and saves a single
`data/processed/metadata.csv` that the classification and detection pipelines load.

Run this notebook FIRST. Then run either of the two downstream notebooks or both in parallel:

- `data-preprocessing-classification.ipynb` → produces `data/processed/classification_data/...`
- `data-preprocessing-detection.ipynb` → produces `data/processed/detection_data/...`


# Mushroom Dataset Preprocessing (Prototype)

Load a small subset of mushroom images from multiple raw datasets, validate, deduplicate, and split into train / val / test for the species classifier.

**Inputs:** `data/raw/<source>/<Species>/<image>.jpg`

**Outputs:**
- `data/processed/classification_data/{train,val,test}/<Species>/*.jpg`
- `data/processed/metadata.csv`
- `data/processed/detection_data/{images,labels}/{split}/<image>.<ext>` (YOLO detector dataset)


## 1. Setup


In [8]:
from pathlib import Path
import hashlib
import random
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import json

import zipfile
import csv
import io
import tempfile
import tarfile
import shutil
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', 20)


def safe(name):
    """Filesystem-friendly name: replace spaces and path separators."""
    return str(name).replace(' ', '_').replace('/', '_').replace(':', '_')

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}


## 2. Paths and configuration

Where data lives, what species to look for, how many images per (species, source) to keep.

Edit these values to suit your project; everything downstream reads from them.


In [9]:
RAW_DIR = Path('data/raw')
PROCESSED_DIR = Path('data/processed')
CLASSIFICATION_DIR = PROCESSED_DIR / 'classification_data'
DETECTION_DIR = PROCESSED_DIR / 'detection_data'

for p in [RAW_DIR, PROCESSED_DIR, CLASSIFICATION_DIR, DETECTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Most commonly photographed/identified species (iconic + common).
TARGET_SPECIES = [
    'Amanita muscaria',
    'Boletus edulis',
    'Cantharellus cibarius',
    'Pleurotus ostreatus',
    'Agaricus bisporus',
    'Laetiporus sulphureus',
    'Morchella esculenta',
    'Coprinus comatus',
]

SUBSET_PER_SPECIES = None   # None = use all gathered images. Set a number to cap per (species, source).


## 3. Download a small sample

Auto-downloads a subset of mushroom images from multiple sources. Each source is independent — if one fails or is skipped, the others still run.

- **iNaturalist** (auto, REST API): up to `PER_SOURCE['inaturalist']` URL downloads per species.
- **GBIF** (auto, REST API — aggregates iNaturalist, Mushroom Observer, and more): up to `PER_SOURCE['gbif']` URL downloads per species.
- **FungiTastic** (manual): no public per-species API. The full ~150 GB set lives on Kaggle. Drop images into `data/raw/fungitastic/<Species>/`; the scan step picks them up automatically.
- **DF20** (auto-extract): fetcher (3.4) writes a per-species wishlist, then a post-process step in 3.5 downloads the ~6.4 GB `DF20-300px.tar.gz` once (cached under `data/raw/_tarballs/`), indexes it, and extracts up to `PER_SOURCE['df20']` matching images per species. Re-runs skip the download when the cache is present.

All downloads land in `data/raw/<source>/<Species>/<file>.jpg`. Re-runs are safe: if a species already has `PER_SOURCE[<source>]` images in its folder, the next run skips it.


### 3.1 iNaturalist

Public REST API (`/v1/observations`), stdlib only. Filters to research-grade observations and returns the `medium`-sized photo URL. Each species returns up to `limit` photos.


In [10]:
import urllib.request
import urllib.parse

def fetch_inaturalist(species, limit=30):
    """Return up to `limit` (url, filename) tuples from iNaturalist for `species`."""
    url = "https://api.inaturalist.org/v1/observations?" + urllib.parse.urlencode({
        "taxon_name": species,
        "quality_grade": "research",
        "per_page": limit,
        "page": 1,
    })
    results = []
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            data = json.loads(r.read().decode())
        for i, obs in enumerate(data.get("results", [])):
            for j, photo in enumerate(obs.get("photos", [])):
                u = (photo.get("url") or "").replace("square", "medium")
                if u:
                    results.append((u, f"inat_{i}_{j}.jpg"))
                if len(results) >= limit:
                    return results
    except Exception:
        pass
    return results


### 3.2 GBIF (Global Biodiversity Information Facility)

Public REST API (`api.gbif.org/v1/occurrence/search`), stdlib only. Filters to occurrences that have `mediaType=StillImage` and returns the image URL from `media[].identifier`. GBIF aggregates iNaturalist, Mushroom Observer, and many other sources, so a single GBIF query often covers multiple underlying databases. Each species returns up to `limit` image URLs.


In [11]:
def fetch_gbif(species, limit=30):
    """Return up to `limit` (url, filename) tuples from GBIF for `species`."""
    url = "https://api.gbif.org/v1/occurrence/search?" + urllib.parse.urlencode({
        "scientificName": species,
        "mediaType": "StillImage",
        "limit": limit,
        "offset": 0,
    })
    results = []
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            data = json.loads(r.read().decode())
        for i, occ in enumerate(data.get("results", [])):
            for j, media in enumerate(occ.get("media", [])):
                if media.get("type") == "StillImage":
                    u = media.get("identifier") or ""
                    if u:
                        ext = ".jpg" if "jpg" in u.lower() or "jpeg" in u.lower() else ".img"
                        results.append((u, f"gbif_{i}_{j}{ext}"))
                if len(results) >= limit:
                    return results
    except Exception:
        pass
    return results


### 3.3 FungiTastic (manual subset)

FungiTastic has no public REST API that supports subset image downloads. The full dataset is ~1.5M images (~150 GB) and is distributed as tarballs via Kaggle (`picekl/fungitastic`) or the official `download.py` script at https://github.com/bohemianvra/FungiTastic.

**To use it:** pick 5-20 images per target species from the FungiTastic website, Kaggle, or Google Images and drop them into `data/raw/fungitastic/<Species>/`. The scan step (4) will pick them up automatically and combine them with the other sources. Re-running 3.5 is a no-op for this source once the folder has enough images.

The fetcher below is informational: it prints clear instructions whenever a target species has fewer than `PER_SOURCE['fungitastic']` images, then returns `[]`. The orchestrator distinguishes this from "already done" in its status line.


In [12]:
def fetch_fungitastic(species, limit=50):
    """FungiTastic: no public subset-download API.

    Returns []. When the folder is short of `limit` images, prints clear
    manual-mode instructions so the user knows what to do. Re-runs are
    cheap: once the folder has enough images, this function is silent.
    """
    out_dir = Path('data/raw/fungitastic') / safe(species)
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = [p for p in out_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
    if len(existing) >= limit:
        return []  # already populated; orchestrator prints 'cached'
    print(f'   \u26a0\ufe0f  FungiTastic/{species}: MANUAL source. Need {limit - len(existing)} more images.')
    print(f'      Drop images into {out_dir}/')
    print('      Or: pip install kaggle && kaggle datasets download -d picekl/fungitastic -p data/raw/_kaggle_cache/')
    print('         (~150 GB, several hours on broadband).')
    return []


def kaggle_setup():
    """Print FungiTastic-via-Kaggle setup instructions."""
    print('To enable Kaggle-based FungiTastic download:')
    print('  1. Generate API token at https://www.kaggle.com/settings')
    print('  2. Place kaggle.json at ~/.kaggle/kaggle.json  (chmod 600)')
    print('  3. pip install kaggle')
    print('  4. kaggle datasets download -d picekl/fungitastic -p data/raw/_kaggle_cache/')
    print('     (~150 GB, several hours). Full set then lives under data/raw/_kaggle_cache/fungitastic/.')
    print('  5. Optionally hand-pick target species into data/raw/fungitastic/<Species>/.')
    print('  6. Re-run this notebook; the scan step will pick them up automatically.')


### 3.4 DF20 (Danish Fungi 2020) — auto-extract from cached tarball

`DF20-300px` is the canonical subset for this pipeline (~6.4 GB compressed, ~24k images at 300px resolution); the full ~110 GB set is overkill for prototype training. Tarball + metadata live at `ptak.felk.cvut.cz`.

DF20 has no per-image HTTP URLs, so we cannot use the URL-download path. Instead the pipeline:

1. **Fetcher (this cell)** downloads `DF20-metadata.zip` once (cached under `tempfile.gettempdir()/DF20-train_metadata.csv`), filters to each target species, and writes `_wishlist_<Species>.json` containing up to `PER_SOURCE['df20']` `image_path` values (e.g. `2238546328-30620.JPG`). Returns `[]` — the orchestrator handles extraction.
2. **Post-process in 3.5**: downloads `DF20-300px.tar.gz` (~6.4 GB) **once** to `data/raw/df20/_tarballs/DF20-300px.tar.gz`. Re-runs reuse the cached tarball. On the first run (or if the index cache is missing), streams the full tar once to build a `{basename → member_name}` index (saved as `DF20-300px.tar.idx.json`). Subsequent runs reuse the index in <1s. For each species' wishlist, opens the tarball and extracts matching members into `data/raw/df20/<Species>/df20_<basename>.<ext>`.

Verified row counts per target species (from `DF20-train_metadata_PROD-2.csv`):

- Amanita muscaria 863  \u2022  Boletus edulis 811  \u2022  Cantharellus cibarius 401  \u2022  Pleurotus ostreatus 931  \u2022  Agaricus bisporus 33  \u2022  Laetiporus sulphureus 613  \u2022  Morchella esculenta 84  \u2022  Coprinus comatus 760

Manual-mode escape hatch (still works): the `_wishlist_*.json` files remain in `data/raw/df20/`. Run `python3 -c "import json,os; [print(p) for p in json.load(open('data/raw/df20/_wishlist_<Species>.json'))]"` to see what would be extracted, then use `tar -xzf data/raw/df20/_tarballs/DF20-300px.tar.gz -C /tmp/df20_extract <member>` per file.


In [13]:
DF20_META_URL = "http://ptak.felk.cvut.cz/plants/DanishFungiDataset/DF20-metadata.zip"
DF20_TAR_URL  = "http://ptak.felk.cvut.cz/plants/DanishFungiDataset/DF20-300px.tar.gz"
DF20_DIR      = Path('data/raw/df20')
DF20_TARBALL_DIR = DF20_DIR / '_tarballs'
DF20_TARBALL_PATH = DF20_TARBALL_DIR / 'DF20-300px.tar.gz'
DF20_CSV_CACHE = Path(tempfile.gettempdir()) / 'DF20-train_metadata.csv'

# Per-species wishlists persist as JSON files; no in-memory coordination needed.


def _df20_ensure_metadata_csv():
    """Download DF20 metadata ZIP if not yet cached locally."""
    if DF20_CSV_CACHE.exists():
        return True
    try:
        DF20_TARBALL_DIR.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(DF20_META_URL, timeout=60) as r:
            zf_bytes = r.read()
        with zipfile.ZipFile(io.BytesIO(zf_bytes)) as zf:
            candidates = [n for n in zf.namelist()
                          if n.endswith('.csv') and 'metadata' in n.lower() and 'MACOSX' not in n]
            if not candidates:
                return False
            csv_name = candidates[0]
            with zf.open(csv_name) as src, open(DF20_CSV_CACHE, 'wb') as dst:
                shutil.copyfileobj(src, dst)
        return True
    except Exception as e:
        print(f'   \u26a0\ufe0f  DF20 metadata fetch failed: {e}')
        return False


def fetch_df20(species, limit=200):
    """Fetch DF20 metadata, build wishlist of image paths matching `species`.

    Returns [] always (no per-image URLs). Writes:
      data/raw/df20/_wishlist_<safe_species>.json   -> list of basenames
    """
    if not _df20_ensure_metadata_csv():
        return []
    species_lc = str(species).strip().lower()
    species_safe = safe(species)
    wishlist_path = DF20_DIR / f'_wishlist_{species_safe}.json'
    DF20_DIR.mkdir(parents=True, exist_ok=True)
    matches = []
    try:
        with open(DF20_CSV_CACHE, encoding='utf-8', errors='replace') as f:
            rdr = csv.DictReader(f)
            for row in rdr:
                # DF20 scientificName includes author citations like
                # 'Amanita muscaria (L.) Lam., 1783' or 'Boletus edulis Bull.',
                # so strict equality never matches. Compare only the first two
                # whitespace-split tokens (genus + specific epithet).
                _name_parts = (row.get('scientificName') or '').strip().lower().split()
                _binomial = ' '.join(_name_parts[:2]) if len(_name_parts) >= 2 else ''
                if _binomial == species_lc:
                    p = (row.get('image_path') or row.get('ImageUniqueID') or '').strip()
                    if p:
                        matches.append(os.path.basename(p))
                if len(matches) >= limit:
                    break
        with open(wishlist_path, 'w') as wf:
            json.dump(matches, wf)
        print(f'   \u2139\ufe0f  DF20/{species}: wishlist built ({len(matches)} basenames)')
    except Exception as e:
        print(f'   \u26a0\ufe0f  DF20 wishlist build failed for {species}: {e}')
        return []
    return []  # post-process in 3.5 does the actual extraction



### 3.5 Run all sources

The orchestrator runs in three phases:

**Phase A — URL-based fetch (parallel).** For each species \u00d7 source, call `fetch_<source>(species, limit=cap)`. Returned `(url, filename)` tuples are downloaded sequentially within the worker via `_download(url, dest)` with `time.sleep(API_DELAY)` between requests. `inaturalist` and `gbif` populate the URL list; `fungitastic` returns `[]` (manual mode, prints instructions when short); `df20` returns `[]` after writing the wishlist JSON.

**Phase B — DF20 cached-tarball extraction (sequential).** For each wishlist, downloads `DF20-300px.tar.gz` once (cached under `data/raw/df20/_tarballs/`), builds a `basename \u2192 member_name` index (cached as `DF20-300px.tar.idx.json`), and extracts matching members into `data/raw/df20/<Species>/`. The whole phase is a no-op when the tarball is cached, the wishlist is empty, or all wishlisted files are already extracted.

**Status messages** distinguish: `cached` (folder already has enough images), `manual/instructions` (FungiTastic, short folder), `downloading` (URL fetch in progress), `(no new)` (URLs exhausted but folder has some images), and `extracting (DF20)` (Phase B is doing work).

Any source whose `fetch_<source>` function is not defined (e.g. its sub-section was not run) is silently skipped — the other sources still complete.


In [ ]:
# Per-source caps. iNat/GBIF are URL-fetched against the cap.
# FungiTastic is informational (no auto-download); its cap is informational.
# DF20 is wishlist-driven; its cap is the wishlist size (post-process extracts up to that many).
PER_SPECIES_DEFAULT = 50
PER_SOURCE = {
    "inaturalist": 100,
    "gbif":         100,
    "fungitastic":   50,
    "df20":         200,
}

API_DELAY = 0.2
MAX_WORKERS = 4

SOURCES = [
    ("inaturalist", Path('data/raw/inaturalist')),
    ("gbif",         Path('data/raw/gbif')),
    ("fungitastic",  Path('data/raw/fungitastic')),
    ("df20",         Path('data/raw/df20')),
]


def _download(url, dest):
    try:
        with urllib.request.urlopen(url, timeout=20) as r, open(dest, 'wb') as f:
            f.write(r.read())
        return True
    except Exception:
        return False


def _download_species_source(species, name, root, fetcher):
    """Phase A worker. Returns (species, name, total, new, status)."""
    if fetcher is None:
        return (species, name, 0, 0, 'no fetcher')
    cap = PER_SOURCE.get(name, PER_SPECIES_DEFAULT)
    sf = safe(species)
    out_dir = Path(root) / sf
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = [p for p in out_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
    if len(existing) >= cap:
        return (species, name, len(existing), 0, 'cached')
    try:
        urls = fetcher(species, limit=cap)
    except Exception:
        urls = []
    if not urls:
        return (species, name, len(existing), 0,
                'manual' if name == 'fungitastic' else 'empty')
    downloaded = 0
    for url, fname in urls:
        if len(existing) >= cap:
            break
        dest = out_dir / fname
        if dest.exists():
            continue
        if _download(url, dest):
            existing.append(dest)
            downloaded += 1
        time.sleep(API_DELAY)
    return (species, name, len(existing), downloaded, 'downloaded')


# ---- Phase A: parallel URL fetches ----
print(f'Downloading from {len(TARGET_SPECIES) * len(SOURCES)} species\u00d7source combos ({MAX_WORKERS} workers)...')
phase_a_results = []
total_new_a = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {}
    for species in TARGET_SPECIES:
        for name, root in SOURCES:
            fetcher = globals().get(f'fetch_{name}')
            futures[ex.submit(_download_species_source, species, name, root, fetcher)] = (species, name)
    for fut in as_completed(futures):
        species, name, total, new, status = fut.result()
        phase_a_results.append((species, name, total, new, status))
        total_new_a += new
        if new:
            label = f'+{new}'
        elif status == 'cached':
            label = f'(already done, {total})'
        elif status == 'manual':
            label = f'(manual \u2014 see instructions above; {total})'
        elif status == 'empty':
            label = f'(no URLs; {total})'
        elif status == 'no fetcher':
            label = '(skipped)'
        else:
            label = f'({status})'
        print(f'  {species:30s} / {name:12s} \u2192 {total:3d} images {label}')

print(f'\nPhase A done \u2014 {total_new_a} new URL-downloaded images.')


# ---- Phase B: DF20 cached-tarball extraction ----
def _df20_post_process(species_list, per_species_cap):
    """Download (once) the DF20 tarball, build a basename index, then extract per-species."""
    print('\nDF20 post-process:')
    DF20_TARBALL_DIR.mkdir(parents=True, exist_ok=True)
    DF20_DIR.mkdir(parents=True, exist_ok=True)

    # Step 1: ensure tarball is present (or skip). The DF20 tarball is ~6.4 GB;
    # without a progress bar a silent download looks frozen on slow networks,
    # so we stream in 4 MB chunks behind a tqdm bar and keep the socket
    # inactivity ceiling at 30 min (not 10) so a brief lull doesn't abort.
    if DF20_TARBALL_PATH.exists() and DF20_TARBALL_PATH.stat().st_size >= 1_000_000_000:
        size_gb = DF20_TARBALL_PATH.stat().st_size / 1e9
        print(f'  Tarball already cached: {DF20_TARBALL_PATH} ({size_gb:.2f} GB)')
    else:
        DF20_TARBALL_PATH.unlink(missing_ok=True)  # drop any partial blob from a prior failed run
        print(f'  Downloading {DF20_TAR_URL} (~6.4 GB) to {DF20_TARBALL_PATH}...')
        print('    (progress bar below; usually 2-8 min on broadband, longer on slow links)')
        try:
            req = urllib.request.Request(DF20_TAR_URL, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=1800) as r:  # 30-min inactivity ceiling per socket
                total = int(r.headers.get('Content-Length') or 0)
                chunk = 4 * 1024 * 1024  # 4 MB
                with open(DF20_TARBALL_PATH, 'wb') as f, tqdm(
                    total=total if total > 0 else None,
                    unit='B', unit_scale=True, unit_divisor=1024,
                    desc='DF20 tarball', mininterval=0.5,
                ) as bar:
                    while True:
                        buf = r.read(chunk)
                        if not buf:
                            break
                        f.write(buf)
                        bar.update(len(buf))
            size_gb = DF20_TARBALL_PATH.stat().st_size / 1e9
            print(f'  Downloaded: {size_gb:.2f} GB')
        except Exception as e:
            print(f'  Download FAILED: {e}. Skipping DF20 extraction.')
            DF20_TARBALL_PATH.unlink(missing_ok=True)  # don't leave a partial blob behind
            return

    # Step 2: build / load basename index (with freshness check).
    # Stale indices point at member names that no longer exist after a
    # tarball re-download, so we verify tarball (size, mtime) before reuse.
    index_path = DF20_TARBALL_PATH.with_suffix('.idx.json')
    tar_stat = DF20_TARBALL_PATH.stat()
    idx = None
    if index_path.exists():
        try:
            with open(index_path) as f:
                cache = json.load(f)
            saved = cache.get('_meta', {}) if isinstance(cache, dict) else {}
            idx_payload = cache.get('index', {}) if isinstance(cache, dict) else None
            if (idx_payload is not None
                    and saved.get('size')  == tar_stat.st_size
                    and abs(saved.get('mtime', 0) - tar_stat.st_mtime) < 1):
                idx = idx_payload
                print(f'  Loaded tarball index ({len(idx)} entries, tarball size/mtime match)')
            else:
                print('  Cached index stale (tarball size or mtime changed) \u2014 re-indexing')
        except Exception:
            idx = None
    if idx is None:
        print('  Indexing tarball members (one-time full stream scan, ~30-90s)...')
        idx = {}
        try:
            with tarfile.open(DF20_TARBALL_PATH, 'r:gz') as tf:
                for m in tqdm(tf, desc='Indexing tarball'):
                    if m.isfile():
                        idx[os.path.basename(m.name)] = m.name
            with open(index_path, 'w') as f:
                json.dump({
                    '_meta': {'size': tar_stat.st_size, 'mtime': tar_stat.st_mtime},
                    'index': idx,
                }, f)
            print(f'  Indexed {len(idx)} members \u2192 {index_path.name}')
        except Exception as e:
            print(f'  Failed to index tarball: {e}. Skipping extraction.')
            return

    # Step 3: one-pass streaming extraction.

    # The previous version did tf.getmember + tf.extractfile per image, but
    # `r:gz` gzipped tarballs do NOT support random access — each non-
    # sequential request retreads huge swaths of the 6 GB gzip stream
    # (~14.7 s per image, ~5 h total for 1,317 images on a healthy disk).
    # Stream the tarball once linearly and write out each match as we see
    # it. Same end state, O(N) cost.
    want_keys: set = set()
    # lowercase basename -> (out_dir, dest_name, species, species_safe)
    want_dest: dict = {}
    picked_per_species: dict = {}
    existing_per_species: dict = {}
    for species in species_list:
        species_safe = safe(species)
        wishlist_path = DF20_DIR / f'_wishlist_{species_safe}.json'
        if not wishlist_path.exists():
            continue
        try:
            with open(wishlist_path) as f:
                wishlist = json.load(f)
        except Exception:
            continue
        if not wishlist:
            existing_per_species[species] = 0
            continue
        out_dir = DF20_DIR / species_safe
        out_dir.mkdir(parents=True, exist_ok=True)
        existing = {p.name for p in out_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS}
        existing_per_species[species] = len(existing)
        picked = 0
        for base in wishlist[:per_species_cap]:
            if picked >= per_species_cap:
                break
            dest_name = f'df20_{base}'
            if dest_name in existing:
                continue
            # DF20-metadata.csv uses uppercase '.JPG' while the tarball stores
            # lowercase '.jpg' — normalize to lowercase for the streaming match.
            k = os.path.basename(base).lower()
            want_keys.add(k)
            want_dest[k] = (out_dir, dest_name, species, species_safe)
            picked += 1
        picked_per_species[species] = picked

    if not want_keys:
        for species in species_list:
            existing_n = existing_per_species.get(species, 0)
            print(f'  {species:30s} → df20  : wishlist satisfied ({existing_n} files present)')
        print(f'\nPhase B done — 0 DF20 images extracted across {len(species_list)} species.')
        return

    total_extracted = 0
    extracted_per_species: dict = {}
    print(f'  Streaming tarball for {len(want_keys)} DF20 matches (one pass)...')
    try:
        with tarfile.open(DF20_TARBALL_PATH, 'r:gz') as tf:
            for m in tqdm(tf, desc='Extracting DF20', mininterval=0.5):
                if not m.isfile():
                    continue
                k = os.path.basename(m.name).lower()
                info = want_dest.pop(k, None)
                if info is None:
                    continue
                out_dir, dest_name, species, _ = info
                dest_path = out_dir / dest_name
                if dest_path.exists():
                    continue
                fh = tf.extractfile(m)
                if fh is None:
                    continue
                with open(dest_path, 'wb') as fout:
                    shutil.copyfileobj(fh, fout)
                total_extracted += 1
                extracted_per_species[species] = extracted_per_species.get(species, 0) + 1
                if not want_dest:
                    break
    except Exception as e:
        print(f'  Failed while streaming tarball: {e}')
        return

    for species in species_list:
        species_safe = safe(species)
        out_dir = DF20_DIR / species_safe
        total = (sum(1 for p in out_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
                 if out_dir.exists() else 0)
        new = extracted_per_species.get(species, 0)
        if new == 0 and picked_per_species.get(species, 0) == 0:
            print(f'  {species:30s} → df20  : wishlist satisfied ({total} files present)')
        elif new == 0:
            print(f'  {species:30s} → df20  : 0 new (folder total: {total})')
        else:
            print(f'  {species:30s} → df20  : extracted {new} images '
                  f'(folder total: {total})')

    n_with_new = sum(1 for n in extracted_per_species.values() if n > 0)
    print(f'\nPhase B done — {total_extracted} DF20 images extracted across {n_with_new} species.')


_df20_post_process(TARGET_SPECIES, PER_SOURCE['df20'])

print('\n\u2705 All sources processed.  Next: run cells 4-7 (metadata, dedup, split, export).')


   ⚠️  FungiTastic/Amanita muscaria: MANUAL source. Need 50 more images.
      Drop images into data/raw/fungitastic/Amanita_muscaria/
      Or: pip install kaggle && kaggle datasets download -d picekl/fungitastic -p data/raw/_kaggle_cache/
         (~150 GB, several hours on broadband).
  Amanita muscaria               / fungitastic  →   0 images (manual — see instructions above; 0)
  Amanita muscaria               / inaturalist  → 100 images (already done, 100)
  Amanita muscaria               / gbif         → 100 images (already done, 100)
  Boletus edulis                 / inaturalist  → 100 images (already done, 100)
  Boletus edulis                 / gbif         → 100 images (already done, 100)
   ⚠️  FungiTastic/Boletus edulis: MANUAL source. Need 50 more images.
      Drop images into data/raw/fungitastic/Boletus_edulis/
      Or: pip install kaggle && kaggle datasets download -d picekl/fungitastic -p data/raw/_kaggle_cache/
         (~150 GB, several hours on broadband).
  Bo

## 4. Scan raw folders and build metadata


In [ ]:
def normalize_species(name):
    if pd.isna(name):
        return None
    name = str(name).strip().replace('_', ' ').replace('-', ' ')
    parts = name.split()
    if len(parts) >= 2:
        return f'{parts[0].capitalize()} {parts[1].lower()}'
    return name

COLUMNS = ['image_path', 'dataset', 'raw_species', 'canonical_species']

rows = []
for img in RAW_DIR.rglob('*'):
    if img.is_dir():
        continue
    if img.suffix.lower() not in IMAGE_EXTS:
        continue
    rows.append({
        'image_path': str(img),
        'dataset': img.relative_to(RAW_DIR).parts[0],
        'raw_species': img.parent.name,
        'canonical_species': normalize_species(img.parent.name),
    })

metadata = pd.DataFrame(rows, columns=COLUMNS)

if len(metadata) == 0:
    print(f'No images found under {RAW_DIR.resolve()}')
    print()
    print('Drop a few images into data/raw/<source>/<Species>/ and re-run this cell.')
    print('For example: data/raw/fungitastic/Amanita_muscaria/IMG_001.jpg')
else:
    print(f'Found {len(metadata)} images across {metadata["dataset"].nunique()} raw datasets')
    print(metadata.groupby('dataset').size().rename('count').to_frame())


Found 1595 images across 2 raw datasets
             count
dataset           
gbif           795
inaturalist    800


## 5. Filter to target species and subset


In [ ]:
metadata = metadata[metadata['canonical_species'].isin(TARGET_SPECIES)].copy()
print(f'After target-species filter: {len(metadata)} images')
print(metadata.groupby(['canonical_species', 'dataset']).size().unstack(fill_value=0))

if SUBSET_PER_SPECIES is not None:
    before = len(metadata)
    metadata = (
        metadata
        .groupby(['canonical_species', 'dataset'], group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), SUBSET_PER_SPECIES), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )
    print(f'After subset ({SUBSET_PER_SPECIES}/pair): {before} -> {len(metadata)}')


After target-species filter: 1595 images
dataset                gbif  inaturalist
canonical_species                       
Agaricus bisporus        95          100
Amanita muscaria        100          100
Boletus edulis          100          100
Cantharellus cibarius   100          100
Coprinus comatus        100          100
Laetiporus sulphureus   100          100
Morchella esculenta     100          100
Pleurotus ostreatus     100          100


## 6. Validate images and deduplicate


In [ ]:
%pip install -q imagehash
import imagehash

# Perceptual-hash dedup catches near-duplicates SHA-256 misses.
# Hamming distance \u2264 5 = visually near-identical.
PHASH_HAMMING_THRESHOLD = 5

def is_valid(path):
    try:
        with Image.open(path) as img:
            img = img.convert('RGB')
            return min(img.size) >= 128
    except Exception:
        return False

metadata['valid'] = [is_valid(p) for p in tqdm(metadata['image_path'], desc='validating')]
metadata = metadata[metadata['valid']].drop(columns=['valid']).reset_index(drop=True)
print(f'After validation: {len(metadata)}')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

metadata['sha256'] = [sha256(p) for p in tqdm(metadata['image_path'], desc='sha256')]
metadata = metadata.drop_duplicates(subset=['sha256']).drop(columns=['sha256']).reset_index(drop=True)
print(f'After SHA-256 dedup: {len(metadata)}')

def phash(path):
    try:
        return imagehash.phash(Image.open(path))
    except Exception:
        return None

hashes = [phash(p) for p in tqdm(metadata['image_path'], desc='phash')]

keep_mask = [True] * len(metadata)
n_phash_dupes = 0
for i in tqdm(range(len(metadata)), desc='phash dedup'):
    if not keep_mask[i] or hashes[i] is None:
        continue
    for j in range(i + 1, len(metadata)):
        if not keep_mask[j] or hashes[j] is None:
            continue
        if (hashes[i] - hashes[j]) <= PHASH_HAMMING_THRESHOLD:  # type: ignore
            keep_mask[j] = False
            n_phash_dupes += 1

metadata = metadata[keep_mask].reset_index(drop=True)
print(f'After perceptual-hash dedup (Hamming \u2264 {PHASH_HAMMING_THRESHOLD}): {len(metadata)} ({n_phash_dupes} near-duplicates removed)')


Note: you may need to restart the kernel to use updated packages.


validating: 100%|██████████| 1595/1595 [00:12<00:00, 129.65it/s]


After validation: 1595


sha256: 100%|██████████| 1595/1595 [00:01<00:00, 1022.12it/s]


After SHA-256 dedup: 1593


phash dedup: 100%|██████████| 1593/1593 [00:01<00:00, 1348.61it/s]

After perceptual-hash dedup (Hamming ≤ 5): 1524 (69 near-duplicates removed)


## 7. Train / val / test split


In [ ]:
train, temp = train_test_split(
    metadata, test_size=0.30, random_state=RANDOM_SEED,
    stratify=metadata['canonical_species'],
)
val, test = train_test_split(
    temp, test_size=0.5, random_state=RANDOM_SEED,
    stratify=temp['canonical_species'],
)
metadata['split'] = ''
metadata.loc[train.index, 'split'] = 'train'
metadata.loc[val.index,   'split'] = 'val'
metadata.loc[test.index,  'split'] = 'test'
print(metadata['split'].value_counts())
print(metadata.groupby(['split', 'canonical_species']).size().unstack(fill_value=0))


split
train    1066
val       229
test      229
Name: count, dtype: int64
canonical_species  Agaricus bisporus  Amanita muscaria  Boletus edulis  \
split                                                                    
test                              20                30              30   
train                             94               139             139   
val                               20                29              30   

canonical_species  Cantharellus cibarius  Coprinus comatus  \
split                                                        
test                                  30                30   
train                                139               139   
val                                   30                30   

canonical_species  Laetiporus sulphureus  Morchella esculenta  \
split                                                           
test                                  29                   30   
train                                138         

## 12. Save metadata


In [ ]:
import datetime

PIPELINE_VERSION = '1.1.0'  # bumped: DF20 cached-tarball extraction now works
metadata['pipeline_version'] = PIPELINE_VERSION

out = PROCESSED_DIR / 'metadata.csv'
out_tmp = PROCESSED_DIR / f'metadata_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
metadata.to_csv(out_tmp, index=False)
out_tmp.rename(out)
print(f'Saved {out} (v{PIPELINE_VERSION}, {len(metadata)} rows)')


Saved data/processed/metadata.csv (v1.1.0, 1524 rows)


## 13. Next steps

- Open `train_classification.ipynb` and point it at `processed/classification_data_aug/` (the augmented dataset, not the raw copies).
- The detection dataset in `processed/detection_data/` uses auto-labeled bboxes from Grounding DINO. Run Step 10c to review low-confidence images and correct them manually before production training.
- Bump `SUBSET_PER_SPECIES` to `None` once the prototype works, to use the full dataset.
